# seu_suite — Colab 실행 노트북
우주 방사선 비트플립 시뮬레이션(2026-09-11 수정본)을 Google Colab에서 돌리는 노트북입니다.

**주의**
- 런타임 유형은 **CPU**로 둡니다 (GPU 불필요, 코드가 CPU 전용).
- 설치는 필요 없습니다 (torch·torchvision·pandas·matplotlib 기본 제공).
- 세션이 끊기면 `/content`가 지워지므로 마지막 셀로 결과를 Drive에 저장하세요.

위에서부터 순서대로 실행합니다. 각 실험 셀은 필요한 것만 골라 실행해도 됩니다.

## 1. 코드 올리기 — 방법 A: zip 직접 업로드 (`seu_suite_v2.zip` 선택)

In [ ]:
from google.colab import files
up = files.upload()   # 파일 선택 창에서 seu_suite_v2.zip 선택
!unzip -o -q seu_suite_v2.zip -d /content
%cd /content/seu_suite
!ls

### (방법 B) Google Drive에 zip을 넣어 두었다면 이 셀을 대신 실행 (`내 드라이브/seu/seu_suite_v2.zip`)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -o -q "/content/drive/MyDrive/seu/seu_suite_v2.zip" -d /content
# %cd /content/seu_suite
# !ls

## 2. 자가진단 — 첫 줄에 `[self-test] bit-flip OK`가 나와야 합니다

In [2]:
!pip install torch torchvision matplotlib pandas

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/124.1 MB 1.2 MB/s eta 0:01:45
   ---------------------------------------- 0.8/124.1 MB 1.2 MB/s eta 0:01:45
   ---------------------------------------- 1.0/124.1 MB 1.2 MB/s eta 0:01:45
   ---------------------------------------- 1.3/124.1 MB 1.2 MB/s eta 0:01:45
    --------------------------------------- 1.6/124.1 MB 1.2 MB/s eta 0:01:44
    --------------------------------------- 1.8/124.1 MB 1.2 MB/s eta 0:01:44
    --------------------------------------- 2.1/124.1 MB 1.2 MB/s eta 0:01:44
    --------------------------------------- 2.4/124.1 MB 1.2 MB/s eta 0:01:44
    --------------------------

In [3]:
!python -c "import torch, torchvision, matplotlib; print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| matplotlib', matplotlib.__version__)"
!python simulate.py --model toy50 --trials 1

torch 2.14.0+cpu | torchvision 0.29.0+cpu | matplotlib 3.10.9
[env] torch=2.14.0+cpu threads=4  data=c:\Users\math\Dropbox\daejin(me)\영재\2026\AI-in-space\data
[env] r=1e-06  train-hours=24.0 (ASSUMPTION, not wall-clock)
[self-test] bit-flip OK
[data] loading toy50 n_train=2000 n_test=500 ...
[model] toy50 n_weight_elements=50  N_bits=1600  n_bias=0  all_params=50
[ckpt] toy50 clean acc=91.80%  0.12s  → c:\Users\math\Dropbox\daejin(me)\영재\2026\AI-in-space\src\results\ckpts\toy50_clean_n2000_e8.pt
  [toy50 infer leo_saa none  1/1] acc=100.00 crash=0 k=0.0 0.00s
[write] c:\Users\math\Dropbox\daejin(me)\영재\2026\AI-in-space\src\results\single_toy50_infer_leo_saa_none_T1_r1e-06.csv
 trial         seed   acc  crashed   k  n_bits  seconds
     0 202692156000 100.0        0 0.0    1600 0.003352
mean_acc=100.00 ± 0.00  median=100.00  survival=1.00  crash=0.00  mean_k=0.00  N_bits=1600  clean_ref=100.00


## 3-1. E1 용량-반응: 확률 p를 바꿔가며 (mnist_linear, 방어 없음 vs clip)
처음 실행 시 MNIST(12MB)를 내려받고 깨끗한 모델을 학습해 `results/ckpts/`에 저장합니다.

In [4]:
for p in [0.0001, 0.001, 0.003, 0.01, 0.03]:
    !python simulate.py --model mnist_linear --mode infer --rad stress --p {p} --defense none --trials 10
    !python simulate.py --model mnist_linear --mode infer --rad stress --p {p} --defense clip --trials 10

[env] torch=2.14.0+cpu threads=4  data=c:\Users\math\Dropbox\daejin(me)\영재\2026\AI-in-space\data
[env] r=1e-06  train-hours=24.0 (ASSUMPTION, not wall-clock)
[self-test] bit-flip OK
[data] loading mnist_linear n_train=60000 n_test=10000 ...
[model] mnist_linear n_weight_elements=101632  N_bits=3252224  n_bias=138  all_params=101770
[ckpt] mnist_linear clean acc=96.66%  1.23s  → c:\Users\math\Dropbox\daejin(me)\영재\2026\AI-in-space\src\results\ckpts\mnist_linear_clean_n60000_e2.pt
  [mnist_linear infer stress none  1/10] acc=96.63 crash=0 k=7.0 0.02s
  [mnist_linear infer stress none  5/10] acc=95.10 crash=0 k=8.0 0.02s
  [mnist_linear infer stress none  10/10] acc=96.66 crash=0 k=8.0 0.02s
[write] c:\Users\math\Dropbox\daejin(me)\영재\2026\AI-in-space\src\results\single_mnist_linear_infer_stress_none_p0.0001.csv
 trial         seed   acc  crashed    k  n_bits  seconds
     0 202613591000 96.63        0  7.0 3252224 0.024449
     1 202613591001 96.63        0 21.0 3252224 0.024228
     2 2


0.3%
0.7%
1.0%
1.3%
1.7%
2.0%
2.3%
2.6%
3.0%
3.3%
3.6%
4.0%
4.3%
4.6%
5.0%
5.3%
5.6%
6.0%
6.3%
6.6%
6.9%
7.3%
7.6%
7.9%
8.3%
8.6%
8.9%
9.3%
9.6%
9.9%
10.2%
10.6%
10.9%
11.2%
11.6%
11.9%
12.2%
12.6%
12.9%
13.2%
13.6%
13.9%
14.2%
14.5%
14.9%
15.2%
15.5%
15.9%
16.2%
16.5%
16.9%
17.2%
17.5%
17.9%
18.2%
18.5%
18.8%
19.2%
19.5%
19.8%
20.2%
20.5%
20.8%
21.2%
21.5%
21.8%
22.1%
22.5%
22.8%
23.1%
23.5%
23.8%
24.1%
24.5%
24.8%
25.1%
25.5%
25.8%
26.1%
26.4%
26.8%
27.1%
27.4%
27.8%
28.1%
28.4%
28.8%
29.1%
29.4%
29.8%
30.1%
30.4%
30.7%
31.1%
31.4%
31.7%
32.1%
32.4%
32.7%
33.1%
33.4%
33.7%
34.0%
34.4%
34.7%
35.0%
35.4%
35.7%
36.0%
36.4%
36.7%
37.0%
37.4%
37.7%
38.0%
38.3%
38.7%
39.0%
39.3%
39.7%
40.0%
40.3%
40.7%
41.0%
41.3%
41.7%
42.0%
42.3%
42.6%
43.0%
43.3%
43.6%
44.0%
44.3%
44.6%
45.0%
45.3%
45.6%
45.9%
46.3%
46.6%
46.9%
47.3%
47.6%
47.9%
48.3%
48.6%
48.9%
49.3%
49.6%
49.9%
50.2%
50.6%
50.9%
51.2%
51.6%
51.9%
52.2%
52.6%
52.9%
53.2%
53.6%
53.9%
54.2%
54.5%
54.9%
55.2%
55.5%
55.9%
56.2%
56.5%
56.

[env] torch=2.14.0+cpu threads=4  data=c:\Users\math\Dropbox\daejin(me)\영재\2026\AI-in-space\data
[env] r=1e-06  train-hours=24.0 (ASSUMPTION, not wall-clock)
[self-test] bit-flip OK
[data] loading mnist_linear n_train=60000 n_test=10000 ...
[model] mnist_linear n_weight_elements=101632  N_bits=3252224  n_bias=138  all_params=101770
[ckpt] loaded c:\Users\math\Dropbox\daejin(me)\영재\2026\AI-in-space\src\results\ckpts\mnist_linear_clean_n60000_e2.pt  stored_acc=96.66%
  [mnist_linear infer stress clip  1/10] acc=96.65 crash=0 k=10.0 0.02s
  [mnist_linear infer stress clip  5/10] acc=96.66 crash=0 k=8.0 0.02s
  [mnist_linear infer stress clip  10/10] acc=96.67 crash=0 k=10.0 0.02s
[write] c:\Users\math\Dropbox\daejin(me)\영재\2026\AI-in-space\src\results\single_mnist_linear_infer_stress_clip_p0.0001.csv
 trial         seed   acc  crashed    k  n_bits  seconds
     0 202664700000 96.65        0 10.0 3252224 0.024551
     1 202664700001 96.65        0 16.0 3252224 0.019972
     2 202664700002 

### 3-1 확장: p를 촘촘하게 (13개) — `!python` 대신 함수를 직접 호출
위 셀은 p 하나마다 `python simulate.py`를 새 프로세스로 띄우므로 매번 torch 임포트·MNIST 로딩·체크포인트 읽기를 반복합니다 (실제 시뮬레이션은 한 trial에 0.03초 정도인데 프로세스 준비에 수 초가 듭니다). 이 셀은 `simulate.run_cell()`을 직접 불러 데이터와 체크포인트를 한 번만 읽고 재사용합니다. 결과 CSV 이름과 형식은 `simulate.py`와 같게 저장하므로 아래 요약표·그래프 셀에 그대로 쓰입니다.

In [5]:
import os, sys, io, time, contextlib, pandas as pd
sys.path.insert(0, "/content/seu_suite")
import simulate as SIM
from presets import Cell

SIM.configure_torch()

P_LIST   = [0.00001, 0.00003, 0.0001, 0.0003, 0.001, 0.002, 0.003, 0.005, 0.01, 0.02, 0.03, 0.05, 0.1]
DEFENSES = ["none", "clip"]
TRIALS   = 10
OUT_DIR  = "results"
os.makedirs(OUT_DIR, exist_ok=True)

data_cache, ckpt_cache = {}, {}      # 데이터·체크포인트를 한 번만 읽고 계속 재사용
t0 = time.time()
for p in P_LIST:
    for d in DEFENSES:
        cell = Cell(model="mnist_linear", mode="infer", rad="stress", defense=d, trials=TRIALS, p=p)
        with contextlib.redirect_stdout(io.StringIO()):          # run_cell의 긴 로그는 숨김
            rows = SIM.run_cell(cell, r=SIM.DEFAULT_R, flip_bias=False,
                                ckpt_dir=SIM.CKPT_DIR_DEFAULT, master_seed=SIM.MASTER_SEED,
                                data_cache=data_cache, ckpt_cache=ckpt_cache, budget_overrides={})
        df = pd.DataFrame(rows)
        path = os.path.join(OUT_DIR, f"single_mnist_linear_infer_stress_{d}_p{p:g}.csv")   # simulate.py와 같은 이름
        df.to_csv(path, index=False)
        s = SIM.make_summary(df).iloc[0]
        print(f"p={p:<8g} {d:<5} n={TRIALS}  median={s['median_acc']:6.2f}  "
              f"survival={s['survival_rate']:.2f}  crash={s['crash_rate']:.2f}   [{time.time()-t0:5.1f}s]")
print(f"done: {len(P_LIST)*len(DEFENSES)} runs, {len(P_LIST)*len(DEFENSES)*TRIALS} trials in {time.time()-t0:.1f}s")

p=1e-05    none  n=10  median= 96.66  survival=1.00  crash=0.00   [  0.3s]
p=1e-05    clip  n=10  median= 96.66  survival=1.00  crash=0.00   [  0.6s]
p=3e-05    none  n=10  median= 96.66  survival=1.00  crash=0.00   [  1.0s]
p=3e-05    clip  n=10  median= 96.66  survival=1.00  crash=0.00   [  1.3s]
p=0.0001   none  n=10  median= 96.63  survival=1.00  crash=0.00   [  1.6s]
p=0.0001   clip  n=10  median= 96.66  survival=1.00  crash=0.00   [  1.8s]
p=0.0003   none  n=10  median= 93.39  survival=0.60  crash=0.00   [  2.1s]
p=0.0003   clip  n=10  median= 96.65  survival=1.00  crash=0.00   [  2.4s]
p=0.001    none  n=10  median= 67.44  survival=0.40  crash=0.00   [  2.6s]
p=0.001    clip  n=10  median= 96.62  survival=1.00  crash=0.00   [  2.9s]
p=0.002    none  n=10  median= 42.19  survival=0.10  crash=0.10   [  3.2s]
p=0.002    clip  n=10  median= 96.49  survival=1.00  crash=0.00   [  3.6s]
p=0.003    none  n=10  median= 40.92  survival=0.00  crash=0.00   [  3.9s]
p=0.003    clip  n=10  me

### 3-1 그래프: 확률 p vs 정확도 (중앙값 + 사분위 범위, 생존률)
위 셀에서 만든 `results/single_mnist_linear_infer_stress_*_p*.csv`를 읽어 p별로 집계합니다. `make_summary`는 p로 구분하지 않으므로 여기서는 (defense, p)로 직접 그룹화합니다.

In [6]:
import glob, pandas as pd, matplotlib.pyplot as plt

files_ = glob.glob("results/single_mnist_linear_infer_stress_*_p*.csv")
df = pd.concat([pd.read_csv(f) for f in files_], ignore_index=True)
df = df[df.single_hit == 0].copy()
df["survived"] = df["acc"] >= 0.9 * df["clean_acc"]   # 깨끗한 정확도의 90% 이상이면 생존

agg = (df.groupby(["defense", "p"])
         .agg(n=("acc", "size"),
              median_acc=("acc", "median"),
              q25=("acc", lambda s: s.quantile(0.25)),
              q75=("acc", lambda s: s.quantile(0.75)),
              survival_rate=("survived", "mean"),
              crash_rate=("crashed", "mean"),
              mean_k=("k", "mean"))
         .reset_index()
         .sort_values(["defense", "p"]))
pd.set_option("display.width", 200)
display(agg)

clean = df["clean_acc"].iloc[0]
colors = {"none": "#C94A24", "clip": "#245EA6"}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))

for d, g in agg.groupby("defense"):
    c = colors.get(d)
    x = g["p"] * 100
    ax1.plot(x, g["median_acc"], "o-", color=c, label=f"{d} (median)")
    ax1.fill_between(x, g["q25"], g["q75"], color=c, alpha=0.18, label=f"{d} (IQR)")
    ax2.plot(x, g["survival_rate"], "s-", color=c, label=d)

ax1.axhline(clean, ls="--", color="grey", lw=1, label=f"clean {clean:.1f}%")
ax1.axhline(0.9 * clean, ls=":", color="grey", lw=1, label="survival line (90% of clean)")
ax1.set_xscale("log"); ax1.set_xlabel("Bit-flip probability p (%)"); ax1.set_ylabel("Accuracy (%)")
ax1.set_ylim(0, 100); ax1.grid(alpha=.3); ax1.legend(fontsize=8)
ax1.set_title("E1 dose-response: accuracy vs p (mnist_linear)")

ax2.set_xscale("log"); ax2.set_xlabel("Bit-flip probability p (%)"); ax2.set_ylabel("Survival rate")
ax2.set_ylim(-0.05, 1.05); ax2.grid(alpha=.3); ax2.legend()
ax2.set_title("E1 survival rate vs p")

plt.tight_layout()
plt.savefig("results/e1_dose_response.png", dpi=150)
plt.show()
print("saved -> results/e1_dose_response.png")

,defense,p,n,median_acc,q25,q75,survival_rate,crash_rate,mean_k
0,clip,0.00001,10,96.660,96.6600,96.6600,1.0,0.0,0.3
1,clip,0.00003,10,96.660,96.6600,96.6600,1.0,0.0,2.2
2,clip,0.00010,10,96.660,96.6525,96.6600,1.0,0.0,10.2
3,clip,0.00030,10,96.645,96.6400,96.6750,1.0,0.0,30.3
4,clip,0.00100,10,96.615,96.4900,96.6700,1.0,0.0,97.4
5,clip,0.00200,10,96.490,96.0350,96.6075,1.0,0.0,197.9
6,clip,0.00300,10,96.400,96.2775,96.5900,1.0,0.0,298.8
7,clip,0.00500,10,96.070,95.8775,96.3700,1.0,0.0,498.7
8,clip,0.01000,10,95.510,94.1350,95.8850,0.9,0.0,1016.3
9,clip,0.02000,10,94.390,93.7050,94.5900,1.0,0.0,2034.9


saved -> results/e1_dose_response.png


C:\Users\math\AppData\Local\Temp\ipykernel_54616\2879586092.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3-2. E2 비트 하나의 운명: 정확히 한 번, 지정한 비트만

In [ ]:
for b in [31, 30, 29, 27, 23, 22, 10, 0]:
    !python simulate.py --model mnist_linear --mode infer --rad stress --single-hit --allowed-bits {b} --trials 30

## 3-3. E3 궤도 임무: 며칠이면 죽나 (leo_saa, T는 하루의 정수배)

In [ ]:
for T in [1, 7, 30, 90]:
    !python simulate.py --model mnist_linear --mode infer --rad leo_saa --T {T} --defense none --trials 10
    !python simulate.py --model mnist_linear --mode infer --rad leo_saa --T {T} --defense clip --trials 10

## 3-4. E5 방어 대결 (toy50, p=5%) — `ensemble`(평균) vs `ensemble_median`(중앙값) 차이에 주목

In [ ]:
for d in ["none", "clip", "tmr", "ensemble", "ensemble_median", "parity"]:
    !python simulate.py --model toy50 --mode infer --rad stress --p 0.05 --defense {d} --trials 20

## 3-5. E4 학습 중 공격 (mnist_linear 1 epoch)

In [ ]:
for d in ["none", "clip"]:
    !python simulate.py --model mnist_linear --mode train --rad stress --p 0.01 --defense {d} --trials 5 --epochs 1

## 3-6. (선택) 논문 전체 표 한 번에 — 20~30분, CIFAR-10 170MB + ResNet 가중치 45MB 다운로드
브라우저 탭을 닫지 마세요.

In [ ]:
# !python simulate.py --preset paper_sweep

## 4. 결과 요약표 — 평균 대신 중앙값·생존률로 보기

In [ ]:
import glob, sys, pandas as pd
sys.path.insert(0, "/content/seu_suite")
from simulate import make_summary

files_ = glob.glob("results/single_*.csv")
df = pd.concat([pd.read_csv(f) for f in files_], ignore_index=True)
summ = make_summary(df)
cols = ["model","mode","rad","defense","allowed_bits","single_hit","p","T_days","n","median_acc","survival_rate","crash_rate","mean_k"]
pd.set_option("display.width", 200)
summ[cols]

## 4-1. 산점도 (13·14주차 방식): 확률 vs 중앙값 정확도

In [ ]:
import matplotlib.pyplot as plt
s = summ[(summ.model=="mnist_linear") & (summ["mode"]=="infer") & (summ.rad=="stress") & (summ.single_hit==0)]
plt.figure(figsize=(8,5))
for d, g in s.groupby("defense"):
    g = g.sort_values("p")
    plt.plot(g["p"]*100, g["median_acc"], "o-", label=d)
plt.xscale("log"); plt.xlabel("Bit-flip probability (%)"); plt.ylabel("Median accuracy (%)")
plt.ylim(0, 100); plt.grid(alpha=.3); plt.legend(); plt.title("E1 dose-response (mnist_linear)")
plt.show()

## 4-2. 비트 지도 (E2): 비트 번호별 생존률

In [ ]:
b = summ[(summ.model=="mnist_linear") & (summ.single_hit==1)].copy()
if len(b):
    b["bit"] = b["allowed_bits"].astype(int)
    b = b.sort_values("bit")
    colors = ["#245EA6" if x==31 else ("#C94A24" if 23<=x<=30 else "#8A97A8") for x in b["bit"]]
    plt.figure(figsize=(9,4))
    plt.bar(b["bit"].astype(str), b["survival_rate"], color=colors)
    plt.ylabel("Survival rate (acc >= 0.9 clean)"); plt.xlabel("Flipped bit (blue=sign, red=exponent, grey=mantissa)")
    plt.title("E2 one-bit criticality (mnist_linear)"); plt.ylim(0,1.05); plt.grid(axis="y", alpha=.3); plt.show()
else:
    print("3-2 셀을 먼저 실행하세요.")

## 5. 결과 저장 — 반드시 실행 (Drive 복사)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/seu/results_$(date +%Y%m%d_%H%M)"
!cp -r results/* "/content/drive/MyDrive/seu/results_$(date +%Y%m%d_%H%M)/"
!ls "/content/drive/MyDrive/seu/" 

### 또는 zip으로 내려받기

In [ ]:
!zip -r -q results.zip results
from google.colab import files
files.download("results.zip")

## 6. (심화) 함수를 직접 불러 쓰기 — 코드 리뷰 문서의 단계와 같은 이름
9주차 실습 2-7의 논문 버전: 깨끗한 모델의 가중치 하나에서 30번 비트만 뒤집고 채점

In [ ]:
import sys, copy, numpy as np, torch
sys.path.insert(0, "/content/seu_suite")
import radiation as R, defenses as D, engine as E, models as M

data = M.load_data_for("mnist_linear", 8000, 2000)
fac = lambda: M.build_model("mnist_linear")
state, clean_acc = E.load_or_train_clean(model_factory=fac, data=data, model_name="mnist_linear",
                                         epochs=2, batch_size=128, lr=0.05, momentum=0.9, ckpt_dir="results/ckpts")
print("clean acc:", clean_acc)

for bit in [30, 23, 0]:
    accs = []
    for seed in range(10):
        m = fac(); m.load_state_dict(copy.deepcopy(state))
        params = R.collect_weight_params(m); cum, n_w = R.weight_bank_meta(params)
        idx = int(np.random.default_rng(seed).integers(0, n_w))
        R.xor_one_bit(params, cum, idx, bit)
        a = E.evaluate(m, data.x_test, data.y_test, "mnist_linear")
        accs.append(a if a is not None else float("nan"))
    print(f"bit {bit:>2}: {np.round(accs, 1)}")